# Fase 0 — Auditoría de Equivalencias de Columnas

**Proyecto:** ProfitMap – Modelo de Riesgo Trimestral SEC EDGAR  
**Entrada:** `datos_sec_edgar/DATASET_MODELO_LISTO.csv` (271,375 × 177)

## Objetivo

Auditar las 177 columnas del dataset y producir las listas de columnas que el siguiente notebook (`1__Risk_Dataset_Builder_v2.ipynb`) debe usar. El deliverable de este notebook son **listas Python listas para copiar y pegar**.

## Filosofía

El pipeline `3_validacion_features.py` ya hizo el trabajo pesado: convirtió los 92 tags XBRL crudos en 73 features `fe_*` normalizadas. Por tanto:

- Los **92 tags `xbrl_raw` se descartan** (redundantes con las `fe_*`).
- Las **73 `fe_*` se clasifican anti-leakage** en tres categorías:

| Categoría | Política | Cuántas |
|---|---|---|
| **A — Prohibidas** | Ingredientes directos del target Z-Score. Nunca como feature. | 9 |
| **B — Autorregresivas** | Sí entran al modelo, pero documentadas. | 4 |
| **C — Libres** | Sin restricción, aportan poder predictivo real. | 57 |
| **Redundantes** | Descartadas (mismo contenido que otra columna). | 3 |

## Pasos

1. Inventario por familia y % de nulos
2. Decisión KEEP / DROP + estrategia de imputación
3. Clasificación anti-leakage A / B / C
4. Verificación del mapeo legacy `RATIO_*` → `fe_ratio_*`
5. Análisis de correlación con target candidato (detectar leakage oculto)
6. **Output final**: listas Python para `Risk_Dataset_Builder_v2`

In [1]:
"""
Fase 0 — Auditoría de Equivalencias de Columnas
================================================
"""
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# Rutas
PROJECT_ROOT = Path('../..').resolve()
DATA_DIR     = PROJECT_ROOT / 'datos_sec_edgar'
DOCS_DIR     = PROJECT_ROOT / 'docs'
DOCS_DIR.mkdir(parents=True, exist_ok=True)
INPUT_CSV    = '../../data_variables_crudas/DATASET_MODELO.csv'

# Carga
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f'Filas: {len(df):,}  |  Columnas: {df.shape[1]}  |  Empresas: {df["cik"].nunique():,}')

# ──────────────────────────────────────────────────────────────
# Paso 1 — Inventario por familia y % nulos
# ──────────────────────────────────────────────────────────────
def clasificar_prefijo(col: str) -> str:
    if col.startswith('fe_ctx_'):    return 'fe_ctx'
    if col.startswith('fe_anual_'):  return 'fe_anual'
    if col.startswith('fe_ratio_'):  return 'fe_ratio'
    if col.startswith('fe_zscore_'): return 'fe_zscore'
    if col.startswith('fe_flag_'):   return 'fe_flag'
    if col.startswith('fe_shares_'): return 'fe_shares'
    if col.startswith('fe_delta_'):  return 'fe_delta'
    if col in {'adsh','cik','name','sic','form','period','filed',
               'fy','fp','fye','countryba','stprba'}:
        return 'meta'
    return 'xbrl_raw'

inv = pd.DataFrame({
    'columna'  : df.columns,
    'familia'  : [clasificar_prefijo(c) for c in df.columns],
    'dtype'    : [str(df[c].dtype) for c in df.columns],
    'pct_nulos': (df.isna().mean() * 100).round(2).values,
})

familia_orden = ['meta','fe_ctx','fe_anual','fe_ratio','fe_zscore',
                 'fe_flag','fe_shares','fe_delta','xbrl_raw']
inv['familia'] = pd.Categorical(inv['familia'], categories=familia_orden, ordered=True)
inv = inv.sort_values(['familia','pct_nulos'],
                      ascending=[True, False]).reset_index(drop=True)

print('\n— Conteo por familia —')
print(inv['familia'].value_counts().sort_index())

Filas: 271,375  |  Columnas: 177  |  Empresas: 11,460

— Conteo por familia —
familia
meta         12
fe_ctx        6
fe_anual      8
fe_ratio     27
fe_zscore     7
fe_flag      10
fe_shares     3
fe_delta     12
xbrl_raw     92
Name: count, dtype: int64


## Pasos 2 y 3 — Decisión KEEP/DROP y clasificación anti-leakage

### Reglas

- **DROP** todos los `xbrl_raw` (92 columnas redundantes con `fe_*`).
- **DROP REDUNDANTES** dentro de `fe_*`:
  - `fe_ctx_revenue_consolidado` → redundante con `fe_anual_revenue`
  - `fe_ctx_periodo` → string redundante con `fy + fp_orden`
  - `fe_anual_factor` → metadato del proceso de anualización, no señal financiera
- **KEEP** las restantes 70 features `fe_*` + 12 de metadata = 82 columnas útiles.

### Estrategia de imputación (ejecutará en Fase 2)

| Estrategia | % nulos | Acción |
|---|---|---|
| `NO_IMPUTAR` | < 5% o metadata | Sin tratamiento |
| `IMP_SIMPLE` | 5–30% | Mediana global o por sector SIC |
| `IMP_CON_FLAG` | 30–70% | Mediana + flag `fe_imputed_<col>` |
| `IMP_ESTRUCTURAL` | ≥ 70% | `0 + flag fe_isnull_<col>` (la ausencia es señal) |

In [2]:
# ──────────────────────────────────────────────────────────────
# Paso 2 — Decisión KEEP/DROP
# ──────────────────────────────────────────────────────────────
DROP_REDUNDANTES = [
    'fe_ctx_revenue_consolidado',  # = fe_anual_revenue
    'fe_ctx_periodo',              # = fy + fp_orden
    'fe_anual_factor',             # metadato del proceso
]

inv['decision'] = 'KEEP'
inv.loc[inv['familia']=='xbrl_raw', 'decision'] = 'DROP_XBRL_RAW'
inv.loc[inv['columna'].isin(DROP_REDUNDANTES), 'decision'] = 'DROP_REDUNDANTE'

# ──────────────────────────────────────────────────────────────
# Paso 3 — Estrategia de imputación
# ──────────────────────────────────────────────────────────────
def estrategia(row):
    if row['decision'] != 'KEEP':  return '—'
    if row['familia'] == 'meta':   return 'NO_IMPUTAR'
    p = row['pct_nulos']
    if p < 5:    return 'NO_IMPUTAR'
    if p < 30:   return 'IMP_SIMPLE'
    if p < 70:   return 'IMP_CON_FLAG'
    return 'IMP_ESTRUCTURAL'

inv['estrategia'] = inv.apply(estrategia, axis=1)

# ──────────────────────────────────────────────────────────────
# Paso 3 — Clasificación anti-leakage A / B / C
# ──────────────────────────────────────────────────────────────
# Categoría A: ingredientes directos del target Z-Score (NO van al modelo)
FEATURES_A_PROHIBIDAS = [
    'fe_zscore_x1_wc_assets',
    'fe_zscore_x2_re_assets',
    'fe_zscore_x3_ebit_assets',
    'fe_zscore_x4_equity_liab',
    'fe_zscore_x5_rev_assets',
    'fe_zscore_altman',
    'fe_zscore_risk_score',
    'fe_flag_altman_distress',
    'fe_flag_altman_grey',
]

# Categoría B: autorregresivas controladas (entran al modelo, documentadas)
FEATURES_B_AUTOREGRESIVAS = [
    'fe_delta_zscore_qoq',
    'fe_delta_risk_score_qoq',
    'fe_delta_risk_score_prev',
    'fe_delta_risk_deterioro',
]

def asignar_categoria(row):
    if row['decision'] != 'KEEP':              return '—'
    if row['familia'] == 'meta':               return 'META'
    if row['columna'] in FEATURES_A_PROHIBIDAS: return 'A_PROHIBIDA'
    if row['columna'] in FEATURES_B_AUTOREGRESIVAS: return 'B_AUTOREGRESIVA'
    return 'C_LIBRE'

inv['categoria'] = inv.apply(asignar_categoria, axis=1)

# ──────────────────────────────────────────────────────────────
# Resúmenes
# ──────────────────────────────────────────────────────────────
print('— Decisión global —')
print(inv['decision'].value_counts())
print()

print('— Estrategia de imputación (solo KEEP) —')
inv_keep = inv[inv['decision']=='KEEP'].copy()
print(inv_keep['estrategia'].value_counts())
print()

print('— Clasificación anti-leakage (solo KEEP) —')
print(inv_keep['categoria'].value_counts())
print()

print('— Detalle de variables IMP_ESTRUCTURAL (≥70% nulos, missing = señal) —')
print(inv_keep[inv_keep['estrategia']=='IMP_ESTRUCTURAL']
      [['columna','familia','pct_nulos','categoria']].to_string(index=False))

— Decisión global —
decision
DROP_XBRL_RAW      92
KEEP               82
DROP_REDUNDANTE     3
Name: count, dtype: int64

— Estrategia de imputación (solo KEEP) —
estrategia
IMP_SIMPLE         39
NO_IMPUTAR         22
IMP_CON_FLAG       18
IMP_ESTRUCTURAL     3
Name: count, dtype: int64

— Clasificación anti-leakage (solo KEEP) —
categoria
C_LIBRE            57
META               12
A_PROHIBIDA         9
B_AUTOREGRESIVA     4
Name: count, dtype: int64

— Detalle de variables IMP_ESTRUCTURAL (≥70% nulos, missing = señal) —
              columna  familia  pct_nulos categoria
 fe_ratio_rnd_revenue fe_ratio    82.0800   C_LIBRE
 fe_ratio_sga_revenue fe_ratio    78.5000   C_LIBRE
fe_ratio_margen_bruto fe_ratio    75.6400   C_LIBRE


## Pasos 4 y 5 — Verificación del mapeo legacy y correlación con target

### Mapeo legacy (informativo, NO se usa en el builder v2)

El builder v1 buscaba 10 ratios con prefijo `RATIO_*`. El pipeline actual los produce con prefijo `fe_ratio_*`. Verificamos que cada destino existe.

### Análisis de correlación

Buscamos features de **Categoría C con |correlación| > 0.85** respecto a `fe_zscore_risk_score`. Si encontramos alguna, hay leakage oculto y debemos moverla a B o A.

In [3]:
# ──────────────────────────────────────────────────────────────
# Paso 4 — Mapeo legacy RATIO_* → fe_ratio_*
# ──────────────────────────────────────────────────────────────
RATIOS_LEGACY_MAPEO = {
    'RATIO_apalancamiento'      : 'fe_ratio_apalancamiento',
    'RATIO_liquidez_corriente'  : 'fe_ratio_liquidez',
    'RATIO_deuda_equity'        : 'fe_ratio_deuda_equity',
    'RATIO_margen_operativo'    : 'fe_ratio_margen_operativo',
    'RATIO_margen_neto'         : 'fe_ratio_margen_neto',
    'RATIO_cobertura_intereses' : 'fe_ratio_cobertura_intereses',
    'RATIO_cash'                : 'fe_ratio_cash',
    'RATIO_ROA'                 : 'fe_ratio_roa',
    'RATIO_ROE'                 : 'fe_ratio_roe',
    'RATIO_cashflow_deuda'      : 'fe_ratio_cashflow_deuda',
}
faltantes = [(k,v) for k,v in RATIOS_LEGACY_MAPEO.items() if v not in df.columns]
assert not faltantes, f'Mapeo legacy roto: {faltantes}'
print(f'✅ Los {len(RATIOS_LEGACY_MAPEO)} ratios legacy mapean correctamente.')

ratios_extra = sorted([c for c in df.columns
                       if c.startswith('fe_ratio_')
                       and c not in RATIOS_LEGACY_MAPEO.values()])
print(f'➕ {len(ratios_extra)} ratios extra disponibles (oportunidad sobre v1):')
for c in ratios_extra:
    print(f'   {c}')

# ──────────────────────────────────────────────────────────────
# Paso 5 — Correlación con target candidato
# ──────────────────────────────────────────────────────────────
TARGET_PROXY = 'fe_zscore_risk_score'
num_cols = [c for c in inv_keep['columna']
            if df[c].dtype.kind in 'fiu' and c != TARGET_PROXY]

corr = (df[num_cols + [TARGET_PROXY]]
        .corr()[TARGET_PROXY].drop(TARGET_PROXY).abs()
        .sort_values(ascending=False))

corr_df = corr.reset_index()
corr_df.columns = ['columna','corr_abs']
corr_df = corr_df.merge(inv_keep[['columna','familia','categoria']],
                        on='columna', how='left')

print('\n— TOP 15 columnas más correlacionadas con risk_score —')
print(corr_df.head(15).to_string(index=False))

# Alerta de leakage oculto
sospechosas = corr_df[(corr_df['categoria']=='C_LIBRE') &
                      (corr_df['corr_abs']>0.85)]
print()
if sospechosas.empty:
    print('✅ Ninguna variable de Categoría C con |corr| > 0.85 → sin leakage oculto.')
else:
    print(f'⚠️  {len(sospechosas)} variables de C con |corr| > 0.85. Considerar mover a B/A:')
    print(sospechosas.to_string(index=False))

✅ Los 10 ratios legacy mapean correctamente.
➕ 17 ratios extra disponibles (oportunidad sobre v1):
   fe_ratio_calidad_ingresos
   fe_ratio_capex_revenue
   fe_ratio_capital_trabajo
   fe_ratio_cash_current
   fe_ratio_cfo_revenue
   fe_ratio_deuda_assets
   fe_ratio_deuda_cp_total
   fe_ratio_ebitda_assets
   fe_ratio_fcf_assets
   fe_ratio_goodwill_assets
   fe_ratio_intangibles_assets
   fe_ratio_margen_bruto
   fe_ratio_quick
   fe_ratio_rnd_revenue
   fe_ratio_rotacion_activos
   fe_ratio_sga_revenue
   fe_ratio_tangibilidad

— TOP 15 columnas más correlacionadas con risk_score —
                    columna  corr_abs   familia       categoria
   fe_delta_risk_score_prev    0.9258  fe_delta B_AUTOREGRESIVA
    fe_flag_altman_distress    0.8357   fe_flag     A_PROHIBIDA
           fe_zscore_altman    0.8061 fe_zscore     A_PROHIBIDA
    fe_flag_margen_negativo    0.5150   fe_flag         C_LIBRE
        fe_flag_altman_grey    0.5112   fe_flag     A_PROHIBIDA
  fe_flag_deficit_acumul

## Paso 6 — Output final: listas para `Risk_Dataset_Builder_v2`

Las siguientes constantes son la **fuente de verdad** del proyecto desde este punto. El siguiente notebook (`1__Risk_Dataset_Builder_v2.ipynb`) debe arrancar copiando estas listas al inicio.

### Cómo se usarán en el builder v2

```
df_raw = pd.read_csv('DATASET_MODELO_LISTO.csv')

# Renombrar ingredientes del target a nombres del builder
df_raw['altman_zscore']  = df_raw['fe_zscore_altman']
df_raw['risk_score_0_1'] = df_raw['fe_zscore_risk_score']

# Construir target (shift -1 por empresa)
df_raw['risk_score_next'] = df_raw.groupby('cik')['risk_score_0_1'].shift(-1)

# Selección final de columnas
FEATURES_MODELO = FEATURES_B_AUTOREGRESIVAS + FEATURES_C_LIBRES   # 4 + 57 = 61
COLS_FINALES   = META_COLS + ['risk_score_0_1'] + FEATURES_MODELO + ['risk_score_next']
df_final = df_raw[COLS_FINALES]
```

In [4]:
# ╔════════════════════════════════════════════════════════════╗
# ║  OUTPUT — Listas para Risk_Dataset_Builder_v2              ║
# ╚════════════════════════════════════════════════════════════╝

# ── Metadata ──────────────────────────────────────────────────
META_COLS = inv_keep.loc[inv_keep['familia']=='meta','columna'].tolist()

# ── Ingredientes del target (uso INTERNO en el builder, no van al modelo) ──
INGREDIENTES_TARGET = ['fe_zscore_altman', 'fe_zscore_risk_score']

# ── Categoría A: PROHIBIDAS (excluidas del modelo, ya definidas arriba) ──
# FEATURES_A_PROHIBIDAS

# ── Categoría B: AUTORREGRESIVAS (al modelo, documentadas) ─
# FEATURES_B_AUTOREGRESIVAS

# ── Categoría C: LIBRES (al modelo, sin restricción) ──────
FEATURES_C_LIBRES = inv_keep.loc[inv_keep['categoria']=='C_LIBRE','columna'].tolist()

# ── Redundantes descartadas ───────────────────────────────────
# DROP_REDUNDANTES (ya definida arriba)

# ── Estrategia de imputación por columna (para Fase 2) ────────
ESTRATEGIA_IMPUTACION = dict(zip(inv_keep['columna'],
                                 inv_keep['estrategia'].astype(str)))

# ── Lista combinada de features que entran al modelo ──────────
FEATURES_MODELO = FEATURES_B_AUTOREGRESIVAS + FEATURES_C_LIBRES

# ── Resumen ──────────────────────────────────────────────────
print('═'*60)
print('OUTPUT FINAL — listas para Risk_Dataset_Builder_v2')
print('═'*60)
print(f'META_COLS                : {len(META_COLS):3d}')
print(f'INGREDIENTES_TARGET      : {len(INGREDIENTES_TARGET):3d}  (uso interno, no al modelo)')
print(f'FEATURES_A_PROHIBIDAS    : {len(FEATURES_A_PROHIBIDAS):3d}  (excluidas del modelo)')
print(f'FEATURES_B_AUTOREGRESIVAS: {len(FEATURES_B_AUTOREGRESIVAS):3d}  (al modelo, documentadas)')
print(f'FEATURES_C_LIBRES        : {len(FEATURES_C_LIBRES):3d}  (al modelo, sin restricción)')
print(f'DROP_REDUNDANTES         : {len(DROP_REDUNDANTES):3d}')
print('─'*60)
print(f'TOTAL FEATURES AL MODELO : {len(FEATURES_MODELO):3d}')
print('═'*60)

# ── Imprimir en formato copy-paste para pegar al builder v2 ──
def _print_lista(nombre, valor):
    if isinstance(valor, dict):
        print(f'\n{nombre} = {{')
        for k, v in valor.items():
            print(f'    {k!r:50s}: {v!r},')
        print('}')
    else:
        print(f'\n{nombre} = [')
        for item in valor:
            print(f'    {item!r},')
        print(']')

print('\n\n# ╔══════════════════════════════════════════════════════════╗')
print('# ║  COPIA TODO LO SIGUIENTE AL INICIO DE                    ║')
print('# ║  notebooks/1__Risk_Dataset_Builder_v2.ipynb              ║')
print('# ╚══════════════════════════════════════════════════════════╝')
_print_lista('META_COLS', META_COLS)
_print_lista('INGREDIENTES_TARGET', INGREDIENTES_TARGET)
_print_lista('FEATURES_A_PROHIBIDAS', FEATURES_A_PROHIBIDAS)
_print_lista('FEATURES_B_AUTOREGRESIVAS', FEATURES_B_AUTOREGRESIVAS)
_print_lista('FEATURES_C_LIBRES', FEATURES_C_LIBRES)
_print_lista('DROP_REDUNDANTES', DROP_REDUNDANTES)
_print_lista('RATIOS_LEGACY_MAPEO', RATIOS_LEGACY_MAPEO)
print('\nFEATURES_MODELO = FEATURES_B_AUTOREGRESIVAS + FEATURES_C_LIBRES')

════════════════════════════════════════════════════════════
OUTPUT FINAL — listas para Risk_Dataset_Builder_v2
════════════════════════════════════════════════════════════
META_COLS                :  12
INGREDIENTES_TARGET      :   2  (uso interno, no al modelo)
FEATURES_A_PROHIBIDAS    :   9  (excluidas del modelo)
FEATURES_B_AUTOREGRESIVAS:   4  (al modelo, documentadas)
FEATURES_C_LIBRES        :  57  (al modelo, sin restricción)
DROP_REDUNDANTES         :   3
────────────────────────────────────────────────────────────
TOTAL FEATURES AL MODELO :  61
════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════╗
# ║  COPIA TODO LO SIGUIENTE AL INICIO DE                    ║
# ║  notebooks/1__Risk_Dataset_Builder_v2.ipynb              ║
# ╚══════════════════════════════════════════════════════════╝

META_COLS = [
    'stprba',
    'sic',
    'countryba',
    'adsh',
    'cik',
    'name',
    'form',
    'period',
    'f

In [5]:
# ──────────────────────────────────────────────────────────────
# Generar reporte ejecutivo (docs/auditoria_columnas_reporte.md)
# ──────────────────────────────────────────────────────────────
reporte = f"""# Reporte de Auditoría de Columnas — Fase 0

**Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}  
**Dataset:** `{INPUT_CSV}` ({len(df):,} filas × {df.shape[1]} columnas, {df['cik'].nunique():,} empresas)

---

## Resumen ejecutivo

| Métrica | Valor |
|---|---|
| Columnas iniciales | {df.shape[1]} |
| Descartadas (xbrl_raw) | {(inv['decision']=='DROP_XBRL_RAW').sum()} |
| Descartadas (redundantes) | {(inv['decision']=='DROP_REDUNDANTE').sum()} |
| Conservadas (KEEP) | {(inv['decision']=='KEEP').sum()} |
| Features al modelo | **{len(FEATURES_MODELO)}** |

## Distribución de las features conservadas

| Categoría | # | Política |
|---|---|---|
| Metadata | {len(META_COLS)} | Identificadores, no features |
| Ingredientes del target (uso interno) | {len(INGREDIENTES_TARGET)} | Solo para construir el target |
| A — Prohibidas | {len(FEATURES_A_PROHIBIDAS)} | Excluidas del modelo (leakage directo) |
| B — Autorregresivas | {len(FEATURES_B_AUTOREGRESIVAS)} | Al modelo, documentadas |
| C — Libres | {len(FEATURES_C_LIBRES)} | Al modelo, sin restricción |

## Estrategia de imputación (Fase 2)

| Estrategia | # cols | Cuándo aplica |
|---|---|---|
{chr(10).join(f"| `{est}` | {n} | — |" for est, n in inv_keep['estrategia'].value_counts().items())}

## Variables redundantes descartadas

| Columna | Razón |
|---|---|
| `fe_ctx_revenue_consolidado` | Redundante con `fe_anual_revenue` |
| `fe_ctx_periodo` | String redundante con `fy + fp_orden` |
| `fe_anual_factor` | Metadato del proceso de anualización, no señal financiera |

## Variables IMP_ESTRUCTURAL (≥70% nulos)

Su missing **no es aleatorio sino estructural** — la ausencia es señal:

| Columna | % nulos | Interpretación del missing |
|---|---|---|
| `fe_ratio_rnd_revenue` | 82.1% | Empresa no reporta R&D porque no tiene R&D (sectorial) |
| `fe_ratio_sga_revenue` | 78.5% | Empresa no separa SGA en sus estados |
| `fe_ratio_margen_bruto` | 75.6% | Empresa no reporta COGS (servicios financieros, REITs) |

**En Fase 2 se imputarán con `0 + flag fe_isnull_*`** (la ausencia es información).

## Mapeo legacy verificado

Los 10 ratios `RATIO_*` que el builder v1 buscaba se mapean correctamente a las `fe_ratio_*` del pipeline actual. **17 ratios extra** del pipeline NO se usaban en v1 — son la principal oportunidad de mejora del modelo.

## Análisis de correlación

Ninguna variable de Categoría C tiene |correlación| > 0.85 con `fe_zscore_risk_score`. **No hay leakage oculto.**

## Siguiente paso

Pasar a **Fase 1 — Refactorización del Dataset Builder**: crear `1__Risk_Dataset_Builder_v2.ipynb` copiando las listas generadas en la Celda 8 de este notebook.
"""


print(f'✓ Reporte generado')
print(reporte)

✓ Reporte generado
# Reporte de Auditoría de Columnas — Fase 0

**Fecha:** 2026-04-25  
**Dataset:** `../../data_variables_crudas/DATASET_MODELO.csv` (271,375 filas × 177 columnas, 11,460 empresas)

---

## Resumen ejecutivo

| Métrica | Valor |
|---|---|
| Columnas iniciales | 177 |
| Descartadas (xbrl_raw) | 92 |
| Descartadas (redundantes) | 3 |
| Conservadas (KEEP) | 82 |
| Features al modelo | **61** |

## Distribución de las features conservadas

| Categoría | # | Política |
|---|---|---|
| Metadata | 12 | Identificadores, no features |
| Ingredientes del target (uso interno) | 2 | Solo para construir el target |
| A — Prohibidas | 9 | Excluidas del modelo (leakage directo) |
| B — Autorregresivas | 4 | Al modelo, documentadas |
| C — Libres | 57 | Al modelo, sin restricción |

## Estrategia de imputación (Fase 2)

| Estrategia | # cols | Cuándo aplica |
|---|---|---|
| `IMP_SIMPLE` | 39 | — |
| `NO_IMPUTAR` | 22 | — |
| `IMP_CON_FLAG` | 18 | — |
| `IMP_ESTRUCTURAL` | 3 | — |

##